# 12b Hazard Index Spatial Cross-Validation

State-blocked cross-validation, geographic stability checks, existing-weight exploration, multi-axis classification and go/no-go decision.

In [1]:

from pathlib import Path
import hashlib, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 140)
plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
ANALYSIS_PATH = TABLE_DIR / 'hazard_index_analysis_dataset.csv'
WEIGHTS_PATH = TABLE_DIR / 'dfsi_monte_carlo_sampled_weights.csv'
for d in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]: d.mkdir(parents=True, exist_ok=True)
RANDOM_SEED=42; BOOTSTRAPS=2000; PRIMARY='cumulative_inundation_area_days_per_km2'

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
RAW_HASHES_BEFORE={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
analysis=pd.read_csv(ANALYSIS_PATH)
weights=pd.read_csv(WEIGHTS_PATH)
assert len(analysis)==657
assert len(weights)==30000
print('Analysis rows:',len(analysis),'states:',analysis['State_Name'].nunique())
print('Weights rows:',len(weights),'columns:',weights.columns.tolist())

def safe_spearman(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.spearmanr(dat.iloc[:,0],dat.iloc[:,1]).statistic)
def safe_kendall(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.kendalltau(dat.iloc[:,0],dat.iloc[:,1]).statistic)
def top10_overlap(x,y):
    dat=pd.DataFrame({'x':x,'y':y}).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)==0: return np.nan,0,0
    k=int(np.ceil(0.10*len(dat))); top_x=set(dat['x'].rank(method='first',ascending=False).nsmallest(k).index); top_y=set(dat['y'].rank(method='first',ascending=False).nsmallest(k).index); inter=len(top_x&top_y); union=len(top_x|top_y)
    return inter/union if union else np.nan,inter,k

# Seeded balanced state folds.
state_counts=analysis.groupby('State_Name').size().sort_values(ascending=False)
rng=np.random.default_rng(RANDOM_SEED)
state_order=pd.DataFrame({'State_Name':state_counts.index,'districts':state_counts.values}); state_order['random_key']=rng.random(len(state_order)); state_order=state_order.sort_values(['districts','random_key'],ascending=[False,True])
fold_loads={i:0 for i in range(5)}; state_to_fold={}
for _,row in state_order.iterrows():
    fold=min(fold_loads,key=lambda k:fold_loads[k]); state_to_fold[row['State_Name']]=fold; fold_loads[fold]+=int(row['districts'])
analysis['cv_fold']=analysis['State_Name'].map(state_to_fold).astype(int)
print('Fold loads:',analysis.groupby('cv_fold').size().to_dict())

# Simplex grid: 0.025 increments plus exact equal weights.
candidates=[]; steps=40
for i in range(steps+1):
    for j in range(steps+1-i):
        k=steps-i-j; candidates.append((i/steps,j/steps,k/steps))
candidates.append((1/3,1/3,1/3))
candidates=pd.DataFrame(candidates,columns=['weight_events','weight_duration','weight_flooded_area']).drop_duplicates()
candidates['evenness_distance']=np.sqrt(((candidates[['weight_events','weight_duration','weight_flooded_area']]-1/3)**2).sum(axis=1))
term_cols=['x_events','x_duration','x_flooded_area']
benchmarks={'learned_hazard_cv':'Learned physical-hazard index','hazard_equal_weight_index':'Equal-weight physical-hazard index','flooded_area_component':'Flooded area alone','published_dfsi_benchmark':'Published DFSI','reconstructed_dfsi_benchmark':'Reconstructed DFSI'}
fold_rows=[]; pred_rows=[]; selected_rows=[]
for fold in sorted(analysis['cv_fold'].unique()):
    train=analysis[analysis['cv_fold']!=fold].copy(); test=analysis[analysis['cv_fold']==fold].copy()
    train_terms=train[term_cols].to_numpy(float); test_terms=test[term_cols].to_numpy(float); y_train=train[PRIMARY].to_numpy(float); y_test=test[PRIMARY].to_numpy(float)
    scores=train_terms @ candidates[['weight_events','weight_duration','weight_flooded_area']].to_numpy(float).T
    train_corrs=[safe_spearman(scores[:,i],y_train) for i in range(scores.shape[1])]
    cand_eval=candidates.copy(); cand_eval['training_spearman']=train_corrs
    best=np.nanmax(cand_eval['training_spearman']); near=cand_eval[cand_eval['training_spearman']>=best-0.001].copy(); selected=near.sort_values(['evenness_distance','weight_events','weight_duration','weight_flooded_area']).iloc[0]
    w=selected[['weight_events','weight_duration','weight_flooded_area']].to_numpy(float); learned_test_score=test_terms@w; train_score=train_terms@w
    selected_rows.append({'fold':fold,'test_districts':len(test),'training_districts':len(train),'test_states':'; '.join(sorted(test['State_Name'].unique())),'training_states':'; '.join(sorted(train['State_Name'].unique())),'weight_events':w[0],'weight_duration':w[1],'weight_flooded_area':w[2],'training_spearman':safe_spearman(train_score,y_train),'best_grid_training_spearman':best,'tie_rule':'Chose most even vector within 0.001 Spearman of best training candidate.'})
    method_scores={'learned_hazard_cv':learned_test_score,'hazard_equal_weight_index':test['hazard_equal_weight_index'].to_numpy(float),'flooded_area_component':test['flooded_area_component'].to_numpy(float),'published_dfsi_benchmark':test['published_dfsi_benchmark'].to_numpy(float),'reconstructed_dfsi_benchmark':test['reconstructed_dfsi_benchmark'].to_numpy(float)}
    for method,score in method_scores.items():
        jac,overlap,topn=top10_overlap(score,y_test)
        fold_rows.append({'fold':fold,'method':method,'method_label':benchmarks[method],'test_districts':len(test),'test_states':'; '.join(sorted(test['State_Name'].unique())),'training_states':'; '.join(sorted(train['State_Name'].unique())),'weight_events':w[0] if method=='learned_hazard_cv' else np.nan,'weight_duration':w[1] if method=='learned_hazard_cv' else np.nan,'weight_flooded_area':w[2] if method=='learned_hazard_cv' else np.nan,'training_spearman_selected_weight':safe_spearman(train_score,y_train) if method=='learned_hazard_cv' else np.nan,'heldout_spearman':safe_spearman(score,y_test),'heldout_kendall':safe_kendall(score,y_test),'heldout_top10_overlap_count':overlap,'heldout_top10_n':topn,'heldout_top10_jaccard':jac})
    for idx,(_,row) in enumerate(test.iterrows()):
        pred_rows.append({'dfsi_record_key':row['dfsi_record_key'],'State_Name':row['State_Name'],'district_name':row['district_name'],'fold':fold,'primary_satellite_outcome':row[PRIMARY],'learned_hazard_oof_score':learned_test_score[idx],'hazard_equal_weight_index':row['hazard_equal_weight_index'],'flooded_area_component':row['flooded_area_component'],'published_dfsi_benchmark':row['published_dfsi_benchmark'],'reconstructed_dfsi_benchmark':row['reconstructed_dfsi_benchmark'],'weight_events':w[0],'weight_duration':w[1],'weight_flooded_area':w[2]})
folds=pd.DataFrame(fold_rows); predictions=pd.DataFrame(pred_rows); selected_weights=pd.DataFrame(selected_rows)
folds.to_csv(TABLE_DIR/'hazard_index_cv_folds.csv',index=False); predictions.to_csv(TABLE_DIR/'hazard_index_cv_predictions.csv',index=False); selected_weights.to_csv(TABLE_DIR/'hazard_index_selected_weights.csv',index=False)
summary_rows=[]
for method,label in benchmarks.items():
    f=folds[folds['method']==method]
    score=predictions['learned_hazard_oof_score'] if method=='learned_hazard_cv' else predictions[method]
    summary_rows.append({'method':method,'method_label':label,'folds':f['fold'].nunique(),'median_heldout_spearman_by_fold':f['heldout_spearman'].median(),'mean_heldout_spearman_by_fold':f['heldout_spearman'].mean(),'min_heldout_spearman_by_fold':f['heldout_spearman'].min(),'max_heldout_spearman_by_fold':f['heldout_spearman'].max(),'overall_out_of_fold_spearman':safe_spearman(score,predictions['primary_satellite_outcome']),'overall_out_of_fold_kendall':safe_kendall(score,predictions['primary_satellite_outcome']),'median_heldout_top10_jaccard':f['heldout_top10_jaccard'].median(),'mean_heldout_top10_jaccard':f['heldout_top10_jaccard'].mean()})
cv_summary=pd.DataFrame(summary_rows); cv_summary.to_csv(TABLE_DIR/'hazard_index_cv_benchmark_summary.csv',index=False)

# Geographic stability.
geo_methods={'learned_hazard_oof_score':'Learned physical-hazard OOF score','hazard_equal_weight_index':'Equal-weight physical-hazard index','flooded_area_component':'Flooded area alone','published_dfsi_benchmark':'Published DFSI','reconstructed_dfsi_benchmark':'Reconstructed DFSI','equal_weight_six_component_log_index':'Equal-weight six-component index','population_component':'Population alone'}
geo=analysis.merge(predictions[['dfsi_record_key','learned_hazard_oof_score']],on='dfsi_record_key',how='left')
within_rows=[]
for method,label in geo_methods.items():
    within_rows.append({'scope':'national','State_Name':'ALL','method':method,'method_label':label,'districts':len(geo),'spearman_correlation':safe_spearman(geo[method],geo[PRIMARY]),'kendall_correlation':safe_kendall(geo[method],geo[PRIMARY])})
    for state,sdf in geo.groupby('State_Name'):
        if len(sdf)>=10:
            within_rows.append({'scope':'within_state','State_Name':state,'method':method,'method_label':label,'districts':len(sdf),'spearman_correlation':safe_spearman(sdf[method],sdf[PRIMARY]),'kendall_correlation':safe_kendall(sdf[method],sdf[PRIMARY])})
within=pd.DataFrame(within_rows); within.to_csv(TABLE_DIR/'hazard_index_within_state_results.csv',index=False)
demean_rows=[]
for method,label in geo_methods.items():
    tmp=geo[['State_Name',method,PRIMARY]].replace([np.inf,-np.inf],np.nan).dropna().copy(); tmp['method_demeaned']=tmp[method]-tmp.groupby('State_Name')[method].transform('mean'); tmp['outcome_demeaned']=tmp[PRIMARY]-tmp.groupby('State_Name')[PRIMARY].transform('mean')
    demean_rows.append({'method':method,'method_label':label,'districts':len(tmp),'states':tmp['State_Name'].nunique(),'state_demeaned_spearman':safe_spearman(tmp['method_demeaned'],tmp['outcome_demeaned']),'state_demeaned_kendall':safe_kendall(tmp['method_demeaned'],tmp['outcome_demeaned'])})
demeaned=pd.DataFrame(demean_rows); demeaned.to_csv(TABLE_DIR/'hazard_index_state_demeaned_results.csv',index=False)
rng=np.random.default_rng(RANDOM_SEED); states=np.array(sorted(geo['State_Name'].unique())); block_rows=[]
for method,label in geo_methods.items():
    vals=[]
    for _ in range(BOOTSTRAPS):
        sampled=rng.choice(states,size=len(states),replace=True); boot=pd.concat([geo[geo['State_Name']==s] for s in sampled],ignore_index=True); vals.append(safe_spearman(boot[method],boot[PRIMARY]))
    vals=np.asarray([v for v in vals if np.isfinite(v)])
    block_rows.append({'method':method,'method_label':label,'states_resampled':len(states),'bootstrap_samples':len(vals),'observed_national_spearman':safe_spearman(geo[method],geo[PRIMARY]),'state_block_bootstrap_median_spearman':float(np.median(vals)),'state_block_bootstrap_ci_low':float(np.percentile(vals,2.5)),'state_block_bootstrap_ci_high':float(np.percentile(vals,97.5))})
block_boot=pd.DataFrame(block_rows); block_boot.to_csv(TABLE_DIR/'hazard_index_state_block_bootstrap.csv',index=False)

# Existing Monte Carlo weight exploration.
term_matrix=analysis[['x_events','x_duration','x_fatalities','x_injured','x_flooded_area','x_population']].to_numpy(float); y=analysis[PRIMARY].to_numpy(float); y_rank=stats.rankdata(y); y_centered=y_rank-y_rank.mean(); y_norm=np.sqrt(np.sum(y_centered**2)); weight_cols=['weight_events','weight_duration','weight_fatalities','weight_injured','weight_flooded_area','weight_population']; W=weights[weight_cols].to_numpy(float); correlations=np.empty(len(W),float); chunk=1000
for start in range(0,len(W),chunk):
    end=min(start+chunk,len(W)); scores=W[start:end]@term_matrix.T; ranks=stats.rankdata(scores,axis=1); ranks=ranks-ranks.mean(axis=1,keepdims=True); denom=np.sqrt((ranks**2).sum(axis=1))*y_norm; correlations[start:end]=(ranks@y_centered)/denom
weights_eval=weights.copy(); weights_eval['satellite_spearman_primary']=correlations
summary=[]
for scenario,sdf in list(weights_eval.groupby('scenario'))+[('all',weights_eval)]:
    summary.append({'scenario':scenario,'samples':len(sdf),'median_spearman':sdf['satellite_spearman_primary'].median(),'p90_spearman':sdf['satellite_spearman_primary'].quantile(.90),'p95_spearman':sdf['satellite_spearman_primary'].quantile(.95),'p99_spearman':sdf['satellite_spearman_primary'].quantile(.99),'maximum_in_sample_spearman':sdf['satellite_spearman_primary'].max(),'notes':'In-sample exploratory evaluation of previous DFSI Monte Carlo weights; not evidence of generalisation.'})
weight_distribution=pd.DataFrame(summary); weight_distribution.to_csv(TABLE_DIR/'hazard_index_existing_weight_correlation_distribution.csv',index=False)
top20=weights_eval.sort_values('satellite_spearman_primary',ascending=False).head(20).copy(); fixed=[]
for idx,row in top20.iterrows():
    w=row[weight_cols].to_numpy(float); fold_corrs=[]
    for fold in sorted(analysis['cv_fold'].unique()):
        test_idx=analysis.index[analysis['cv_fold']==fold]; score=term_matrix[test_idx]@w; fold_corrs.append(safe_spearman(score,analysis.loc[test_idx,PRIMARY]))
    fixed.append({'original_row_index':idx,'fixed_fold_mean_heldout_spearman':float(np.nanmean(fold_corrs)),'fixed_fold_min_heldout_spearman':float(np.nanmin(fold_corrs)),'fixed_fold_max_heldout_spearman':float(np.nanmax(fold_corrs))})
top20=top20.reset_index().rename(columns={'index':'original_row_index'}).merge(pd.DataFrame(fixed),on='original_row_index',how='left'); top20.to_csv(TABLE_DIR/'hazard_index_top_existing_weights.csv',index=False)
patterns=[]
for scenario,sdf in list(weights_eval.groupby('scenario'))+[('all',weights_eval)]:
    threshold=sdf['satellite_spearman_primary'].quantile(.99); sdf_top=sdf[sdf['satellite_spearman_primary']>=threshold]
    for col in weight_cols:
        patterns.append({'scenario':scenario,'component_weight':col,'spearman_weight_vs_satellite_correlation':safe_spearman(sdf[col],sdf['satellite_spearman_primary']),'mean_weight_all_samples':sdf[col].mean(),'mean_weight_top_1pct':sdf_top[col].mean(),'top_1pct_samples':len(sdf_top),'notes':'Association is descriptive within saved Monte Carlo samples, not causal importance.'})
patterns=pd.DataFrame(patterns); patterns.to_csv(TABLE_DIR/'hazard_index_component_weight_patterns.csv',index=False)

# Multi-axis classification.
cv_lookup=cv_summary.set_index('method')['overall_out_of_fold_spearman'].to_dict(); learned_oof=cv_lookup.get('learned_hazard_cv',np.nan); equal_oof=cv_lookup.get('hazard_equal_weight_index',np.nan); weight_ranges=selected_weights[['weight_events','weight_duration','weight_flooded_area']].max()-selected_weights[['weight_events','weight_duration','weight_flooded_area']].min(); use_learned=bool(np.isfinite(learned_oof) and np.isfinite(equal_oof) and learned_oof>equal_oof+0.03 and weight_ranges.max()<=0.50); hazard_col='learned_hazard_oof_score' if use_learned else 'hazard_equal_weight_index'; hazard_label='learned out-of-fold physical-hazard score' if use_learned else 'equal-weight physical-hazard index'
classified=geo.copy(); classified['physical_hazard_dimension']=classified[hazard_col]; classified['exposure_dimension']=classified['exposure_score']; classified['human_consequence_dimension']=classified['human_consequence_score']
for col in ['physical_hazard_dimension','exposure_dimension','human_consequence_dimension']:
    classified[col+'_percentile']=classified[col].rank(method='average',pct=True)*100
def category(row):
    h=row['physical_hazard_dimension_percentile']; e=row['exposure_dimension_percentile']; c=row['human_consequence_dimension_percentile']
    if h>=75 and e>=75 and c>=75: return 'high hazard, high exposure and high consequences'
    if h>=75 and c<75: return 'high hazard but lower recorded consequences'
    if c>=75 and h<75: return 'high consequences despite moderate hazard'
    if e>=75 and 40<=h<75 and c<75: return 'high exposure with moderate hazard'
    if h<40 and e<40 and c<40: return 'lower across all three dimensions'
    return 'objective-dependent or mixed profile'
classified['classification_method']=hazard_label; classified['profile_category']=classified.apply(category,axis=1)
classification_cols=['State_Name','district_name','boundary_shape_id','published_dfsi_benchmark','reconstructed_dfsi_benchmark',PRIMARY,'physical_hazard_dimension','physical_hazard_dimension_percentile','exposure_dimension','exposure_dimension_percentile','human_consequence_dimension','human_consequence_dimension_percentile','profile_category','classification_method']
classified[classification_cols].to_csv(TABLE_DIR/'hazard_index_multi_axis_classification.csv',index=False)
profile_summary=classified['profile_category'].value_counts().rename_axis('profile_category').reset_index(name='districts')

# Figures.
fig,ax=plt.subplots(figsize=(9,5.2),constrained_layout=True)
for method in ['learned_hazard_cv','hazard_equal_weight_index','flooded_area_component','published_dfsi_benchmark','reconstructed_dfsi_benchmark']:
    f=folds[folds['method']==method].sort_values('fold'); ax.plot(f['fold'],f['heldout_spearman'],marker='o',label=benchmarks[method])
ax.axhline(0,color='black',linewidth=.8); ax.set_xlabel('State-blocked fold'); ax.set_ylabel('Held-out Spearman correlation'); ax.set_title('Held-out performance by state-blocked fold'); ax.legend(fontsize=8); fig.savefig(FIGURE_DIR/'hazard_index_cv_performance.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(8.5,4.8),constrained_layout=True); selected_weights.set_index('fold')[['weight_events','weight_duration','weight_flooded_area']].plot(kind='bar',stacked=True,ax=ax,color=['#4c78a8','#f58518','#54a24b']); ax.set_ylabel('Selected weight'); ax.set_xlabel('Fold'); ax.set_title('Selected physical-hazard weights by training fold'); ax.legend(['Flood events','Mean duration','Flooded area'],fontsize=8); fig.savefig(FIGURE_DIR/'hazard_index_selected_weights.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.6,5.2),constrained_layout=True); ax.scatter(predictions['learned_hazard_oof_score'],predictions['primary_satellite_outcome'],s=22,alpha=.7); ax.set_xlabel('Out-of-fold learned hazard score'); ax.set_ylabel('Cumulative inundation area-days per km2'); sp=safe_spearman(predictions['learned_hazard_oof_score'],predictions['primary_satellite_outcome']); ax.set_title('Out-of-fold hazard score versus satellite inundation'); ax.text(.02,.95,f'n = {len(predictions)}\nSpearman = {sp:.3f}',transform=ax.transAxes,va='top',fontsize=9); fig.savefig(FIGURE_DIR/'hazard_index_out_of_fold_scatter.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(9,5.2),constrained_layout=True); profile_summary.sort_values('districts').plot.barh(x='profile_category',y='districts',ax=ax,legend=False,color='#4c78a8'); ax.set_xlabel('Districts'); ax.set_ylabel(''); ax.set_title('Descriptive multi-axis district profiles'); ax.text(.01,.02,'High = percentile >= 75; lower = all dimensions < 40. Categories are descriptive, not policy thresholds.',transform=ax.transAxes,fontsize=8); fig.savefig(FIGURE_DIR/'hazard_index_multi_axis_profiles.png',dpi=220,bbox_inches='tight'); plt.close(fig)

# Go/no-go report.
bench_primary=pd.read_csv(TABLE_DIR/'hazard_index_benchmark_results.csv').query('satellite_outcome == @PRIMARY').set_index('index_measure'); paired=pd.read_csv(TABLE_DIR/'hazard_index_paired_bootstrap_comparisons.csv'); learned_summary=cv_summary.set_index('method').loc['learned_hazard_cv']; equal_summary=cv_summary.set_index('method').loc['hazard_equal_weight_index']; floodarea_summary=cv_summary.set_index('method').loc['flooded_area_component']; published_summary=cv_summary.set_index('method').loc['published_dfsi_benchmark']; secondary=pd.read_csv(TABLE_DIR/'hazard_index_benchmark_results.csv'); hsec=secondary[secondary['index_measure']=='hazard_equal_weight_index']; psec=secondary[secondary['index_measure']=='published_dfsi_benchmark']; secjoin=hsec[['satellite_outcome','spearman_correlation']].merge(psec[['satellite_outcome','spearman_correlation']],on='satellite_outcome',suffixes=('_hazard_equal','_published_dfsi')); secjoin['hazard_minus_published']=secjoin['spearman_correlation_hazard_equal']-secjoin['spearman_correlation_published_dfsi']
hazard_vs_published=float(bench_primary.loc['hazard_equal_weight_index','spearman_correlation']-bench_primary.loc['published_dfsi_benchmark','spearman_correlation']); cv_learned_minus_published=float(learned_summary['overall_out_of_fold_spearman']-published_summary['overall_out_of_fold_spearman']); cv_learned_minus_flooded=float(learned_summary['overall_out_of_fold_spearman']-floodarea_summary['overall_out_of_fold_spearman'])
if hazard_vs_published>0.03 and cv_learned_minus_published>0.03 and cv_learned_minus_flooded>=0 and weight_ranges.max()<=0.50 and (secjoin['hazard_minus_published']>0).sum()>=2:
    decision='GO — rebuild the paper'
elif hazard_vs_published>0 or profile_summary['profile_category'].nunique()>=4:
    decision='GO WITH A NARROWER CLAIM'
else:
    decision='NO-GO'
report=f"""# Hazard Index Go/No-Go Report

## Decision

**{decision}**

## Analysis Scope

- Matched districts analysed: {len(analysis)}
- States represented: {analysis['State_Name'].nunique()}
- Primary satellite outcome: `{PRIMARY}`
- Satellite period: Global Flood Database mapped events, 2000-2018, partial temporal overlap with DFSI.
- DFSI/GFD matches: unique accepted matches only.

## Benchmark Correlations: Primary Outcome

- Equal-weight physical-hazard index Spearman: {bench_primary.loc['hazard_equal_weight_index','spearman_correlation']:.4f} [{bench_primary.loc['hazard_equal_weight_index','bootstrap_spearman_ci_low']:.4f}, {bench_primary.loc['hazard_equal_weight_index','bootstrap_spearman_ci_high']:.4f}]
- Published DFSI Spearman: {bench_primary.loc['published_dfsi_benchmark','spearman_correlation']:.4f} [{bench_primary.loc['published_dfsi_benchmark','bootstrap_spearman_ci_low']:.4f}, {bench_primary.loc['published_dfsi_benchmark','bootstrap_spearman_ci_high']:.4f}]
- Reconstructed DFSI Spearman: {bench_primary.loc['reconstructed_dfsi_benchmark','spearman_correlation']:.4f} [{bench_primary.loc['reconstructed_dfsi_benchmark','bootstrap_spearman_ci_low']:.4f}, {bench_primary.loc['reconstructed_dfsi_benchmark','bootstrap_spearman_ci_high']:.4f}]
- Flooded-area component alone Spearman: {bench_primary.loc['flooded_area_component','spearman_correlation']:.4f}
- Population component alone Spearman: {bench_primary.loc['population_component','spearman_correlation']:.4f}

## Paired Bootstrap Interpretation

See `outputs/tables/hazard_index_paired_bootstrap_comparisons.csv`. Paired bootstrap samples use identical district resamples for each method pair. Small point-estimate differences are not treated as decisive unless uncertainty supports them.

## State-Blocked Cross-Validation

- Learned physical-hazard OOF Spearman: {learned_summary['overall_out_of_fold_spearman']:.4f}
- Equal-weight physical-hazard OOF Spearman: {equal_summary['overall_out_of_fold_spearman']:.4f}
- Flooded-area alone OOF Spearman: {floodarea_summary['overall_out_of_fold_spearman']:.4f}
- Published DFSI OOF Spearman: {published_summary['overall_out_of_fold_spearman']:.4f}
- Selected weight medians: events {selected_weights['weight_events'].median():.3f}, duration {selected_weights['weight_duration'].median():.3f}, flooded area {selected_weights['weight_flooded_area'].median():.3f}
- Selected weight ranges: events {selected_weights['weight_events'].min():.3f}-{selected_weights['weight_events'].max():.3f}, duration {selected_weights['weight_duration'].min():.3f}-{selected_weights['weight_duration'].max():.3f}, flooded area {selected_weights['weight_flooded_area'].min():.3f}-{selected_weights['weight_flooded_area'].max():.3f}

## Geographic Stability

- State-demeaned results: see `outputs/tables/hazard_index_state_demeaned_results.csv`.
- State-block bootstrap results: see `outputs/tables/hazard_index_state_block_bootstrap.csv`.
- These checks test whether national correlations are driven only by broad between-state differences.

## Existing Monte Carlo Weight Exploration

- Existing six-component weight samples were evaluated in-sample against the primary satellite outcome.
- This is exploratory and potentially overfit.
- The highest-correlating sampled vector is not called the best index and is not used to define the final framework.

## Multi-Axis Classification

- Physical hazard dimension used: {hazard_label}
- Exposure dimension: transformed population.
- Human-consequence dimension: equal-weight transformed fatalities and injuries.
- Categories are descriptive percentile profiles, not official policy thresholds.

## Scientific Limitations

- GFD covers 2000-2018 and is not a complete inventory of every Indian flood.
- The satellite outcome measures physical inundation; DFSI combines hazard, exposure and recorded human consequences.
- The current chunked GFD aggregation does not provide a valid unique-pixel `ever_flooded_area`; this field is excluded.
- Boundary vintage mismatch and unresolved district matches remain.
- State-blocked folds have only five held-out partitions, so fold-specific estimates are noisy.
- Optimised in-sample Monte Carlo weights are not evidence of generalisation.
"""
(REPORT_DIR/'hazard_index_go_no_go.md').write_text(report,encoding='utf-8')

for path in [TABLE_DIR/'hazard_index_cv_folds.csv',TABLE_DIR/'hazard_index_cv_predictions.csv',TABLE_DIR/'hazard_index_selected_weights.csv',TABLE_DIR/'hazard_index_cv_benchmark_summary.csv',TABLE_DIR/'hazard_index_within_state_results.csv',TABLE_DIR/'hazard_index_state_demeaned_results.csv',TABLE_DIR/'hazard_index_state_block_bootstrap.csv',TABLE_DIR/'hazard_index_existing_weight_correlation_distribution.csv',TABLE_DIR/'hazard_index_top_existing_weights.csv',TABLE_DIR/'hazard_index_component_weight_patterns.csv',TABLE_DIR/'hazard_index_multi_axis_classification.csv',FIGURE_DIR/'hazard_index_cv_performance.png',FIGURE_DIR/'hazard_index_selected_weights.png',FIGURE_DIR/'hazard_index_out_of_fold_scatter.png',FIGURE_DIR/'hazard_index_multi_axis_profiles.png',REPORT_DIR/'hazard_index_go_no_go.md']:
    assert path.exists(), path
assert len(predictions)==len(analysis)
assert set(predictions['fold'])==set(range(5))
assert np.allclose(selected_weights[['weight_events','weight_duration','weight_flooded_area']].sum(axis=1),1.0)
assert (selected_weights[['weight_events','weight_duration','weight_flooded_area']]>=-1e-12).all().all()
RAW_HASHES_AFTER={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
assert RAW_HASHES_BEFORE==RAW_HASHES_AFTER, 'data/raw changed during hazard spatial CV.'
print(report)
display(selected_weights[['fold','test_districts','weight_events','weight_duration','weight_flooded_area','training_spearman']])
display(cv_summary[['method_label','overall_out_of_fold_spearman','overall_out_of_fold_kendall','median_heldout_spearman_by_fold','min_heldout_spearman_by_fold','max_heldout_spearman_by_fold']])
display(weight_distribution)
display(profile_summary)


Analysis rows: 657 states: 35
Weights rows: 30000 columns: ['scenario', 'simulation', 'weight_events', 'weight_duration', 'weight_fatalities', 'weight_injured', 'weight_flooded_area', 'weight_population', 'weight_sum']
Fold loads: {0: 132, 1: 132, 2: 131, 3: 131, 4: 131}


# Hazard Index Go/No-Go Report

## Decision

**GO — rebuild the paper**

## Analysis Scope

- Matched districts analysed: 657
- States represented: 35
- Primary satellite outcome: `cumulative_inundation_area_days_per_km2`
- Satellite period: Global Flood Database mapped events, 2000-2018, partial temporal overlap with DFSI.
- DFSI/GFD matches: unique accepted matches only.

## Benchmark Correlations: Primary Outcome

- Equal-weight physical-hazard index Spearman: 0.4529 [0.3840, 0.5200]
- Published DFSI Spearman: 0.3575 [0.2774, 0.4300]
- Reconstructed DFSI Spearman: 0.3721 [0.2964, 0.4441]
- Flooded-area component alone Spearman: 0.3745
- Population component alone Spearman: 0.3944

## Paired Bootstrap Interpretation

See `outputs/tables/hazard_index_paired_bootstrap_comparisons.csv`. Paired bootstrap samples use identical district resamples for each method pair. Small point-estimate differences are not treated as decisive unless uncertainty supports them.

## State-Blocked Cross-Vali

,fold,test_districts,weight_events,weight_duration,weight_flooded_area,training_spearman
0,0,132,0.150,0.500,0.350,0.426694
1,1,132,0.150,0.450,0.400,0.506496
2,2,131,0.175,0.475,0.350,0.446075
3,3,131,0.175,0.450,0.375,0.532246
4,4,131,0.075,0.425,0.500,0.527893


,method_label,overall_out_of_fold_spearman,overall_out_of_fold_kendall,median_heldout_spearman_by_fold,min_heldout_spearman_by_fold,max_heldout_spearman_by_fold
0,Learned physical-hazard index,0.487375,0.343251,0.432860,0.099192,0.646179
1,Equal-weight physical-hazard index,0.452920,0.323931,0.408415,0.242273,0.606911
2,Flooded area alone,0.374484,0.255994,0.350936,-0.283521,0.589306
3,Published DFSI,0.357479,0.248589,0.394662,0.061987,0.513705
4,Reconstructed DFSI,0.372106,0.259155,0.404098,0.088603,0.537030


,scenario,samples,median_spearman,p90_spearman,p95_spearman,p99_spearman,maximum_in_sample_spearman,notes
0,moderate,10000,0.370688,0.426877,0.442261,0.469139,0.509050,In-sample exploratory evaluation of previous D...
1,narrow,10000,0.371696,0.404282,0.412842,0.429985,0.461547,In-sample exploratory evaluation of previous D...
2,wide,10000,0.367820,0.455607,0.473658,0.505954,0.554843,In-sample exploratory evaluation of previous D...
3,all,30000,0.370724,0.428640,0.449626,0.487261,0.554843,In-sample exploratory evaluation of previous D...


,profile_category,districts
0,objective-dependent or mixed profile,279
1,high consequences despite moderate hazard,113
2,high hazard but lower recorded consequences,113
3,lower across all three dimensions,95
4,"high hazard, high exposure and high consequences",35
5,high exposure with moderate hazard,22
